In [6]:
import math
from typing import List, Dict, Any

In [7]:
# Simple geometric helper functions
def dot_product(v1: List[float], v2: List[float]) -> float:
    return sum(a * b for a, b in zip(v1, v2))

def magnitude(v: List[float]) -> float:
    return math.sqrt(sum(x**2 for x in v))

def cosine_similarity(v1: List[float], v2: List[float]) -> float:
    mag1, mag2 = magnitude(v1), magnitude(v2)
    return dot_product(v1, v2) / (mag1 * mag2) if mag1 and mag2 else 0.0

In [8]:
# --- Simulated Production Vector Database ---
class MockVectorDatabase:
    """
    Simulates a partitioned Vector DB Index structure.
    Instead of checking every record linearly, it clusters data into specific regions.
    """
    def __init__(self):
        # We define two operational index clusters (Voronoi region centroids)
        self.centroids = {
            "CLUSTER_TECH": [0.90, 0.10],
            "CLUSTER_BIO":  [0.10, 0.90]
        }
        # The underlying database storage bucket segregated by cluster keys
        self.storage: Dict[str, List[Dict[str, Any]]] = {
            "CLUSTER_TECH": [],
            "CLUSTER_BIO": []
        }

    def insert_document(self, text: str, embedding: List[float]):
        """Determines the closest cluster centroid and saves the record there."""
        best_cluster = None
        highest_score = -1.0
        
        for cluster_id, centroid_vector in self.centroids.items():
            score = cosine_similarity(embedding, centroid_vector)
            if score > highest_score:
                highest_score = score
                best_cluster = cluster_id
                
        # Append record along with its text data to the isolated cluster space
        self.storage[best_cluster].append({"text": text, "vector": embedding})
        print(f"[DB Write] Saved item to storage room: {best_cluster}")

    def approximate_nearest_neighbor_search(self, query_vector: List[float]) -> List[Dict[str, Any]]:
        """Finds the closest cluster first, then searches ONLY within that cluster."""
        # Step 1: Find the target cluster neighborhood
        target_cluster = None
        highest_centroid_score = -1.0
        for cluster_id, centroid_vector in self.centroids.items():
            score = cosine_similarity(query_vector, centroid_vector)
            if score > highest_centroid_score:
                highest_centroid_score = score
                target_cluster = cluster_id
                
        print(f"[DB Query] Target identified! Searching ONLY inside: {target_cluster}")
        print(f"[DB Query] Skipping alternative rooms entirely. Saved search overhead.")
        
        # Step 2: Rank items only within that chosen cluster array
        results = []
        for item in self.storage[target_cluster]:
            sim = cosine_similarity(query_vector, item["vector"])
            results.append({"text": item["text"], "score": round(sim, 4)})
            
        # Sort results descending by score
        results.sort(key=lambda x: x["score"], reverse=True)
        return results

In [9]:
# --- Initialize and Populate the Database ---
vdb = MockVectorDatabase()

# Populate data points
vdb.insert_document("Tutorial on compiling C++ modules.", [0.93, 0.08])
vdb.insert_document("Building scalable cloud microservices.", [0.88, 0.15])
vdb.insert_document("Understanding cellular mitosis and DNA.", [0.05, 0.95])
vdb.insert_document("Replication methods of marine viral cells.", [0.12, 0.89])

print("\n" + "-"*50 + "\n")

[DB Write] Saved item to storage room: CLUSTER_TECH
[DB Write] Saved item to storage room: CLUSTER_TECH
[DB Write] Saved item to storage room: CLUSTER_BIO
[DB Write] Saved item to storage room: CLUSTER_BIO

--------------------------------------------------



In [10]:
# --- Execute Query ---
# A user query vector focused entirely on a technology theme
user_query_vector = [0.91, 0.11] 

search_matches = vdb.approximate_nearest_neighbor_search(user_query_vector)

print("\nFinal Closest Matches Found:")
for rank, match in enumerate(search_matches, start=1):
    print(f"  Rank {rank}: [Score: {match['score']}] -> \"{match['text']}\"")

[DB Query] Target identified! Searching ONLY inside: CLUSTER_TECH
[DB Query] Skipping alternative rooms entirely. Saved search overhead.

Final Closest Matches Found:
  Rank 1: [Score: 0.9994] -> "Tutorial on compiling C++ modules."
  Rank 2: [Score: 0.9988] -> "Building scalable cloud microservices."
